In [ ]:
!pip install -q transformers datasets accelerate

In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSeq2SeqLM,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
import torch
import re
from tqdm import tqdm

In [ ]:
gsm8k = load_dataset("gsm8k", "main", split="test[:50]")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

In [ ]:
gsm8k[0]["question"]

"Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?"

In [ ]:
gsm8k[0]["answer"]

'Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.\nShe makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.\n#### 18'

#### Few shot prompts

In [ ]:
few_shots = """Q: Emily has 3 apples. Her friend gives her 2 more. How many apples does Emily have now?
A: Emily starts with 3 apples. Her friend gives her 2 more. So, 3 + 2 = 5. The answer is 5.

Q: A pen costs 2 dollars. John buys 4 pens. How much does he pay?
A: Each pen costs 2 dollars. John buys 4 pens. So, 2 × 4 = 8. The answer is 8.

Q: Jake read 5 pages on Monday and 7 pages on Tuesday. How many pages did he read in total?
A: Jake read 5 pages on Monday and 7 on Tuesday. So, 5 + 7 = 12. The answer is 12.
"""

#### Extracting final answer as a single numerical value

In [ ]:
def extract_final_answer(text):
    text = text.replace(",", "")
    matches = re.findall(r"\d+(?:\.\d+)?", text)
    if matches:
        return matches[-1]
    return None

#### Loading specific model - seq2seq or casual LM model

In [ ]:
def load(model_id, use_4bit = True):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    tokenizer = AutoTokenizer.from_pretrained(model_id)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    config = AutoConfig.from_pretrained(model_id)

    if config.is_encoder_decoder:
        model = AutoModelForSeq2SeqLM.from_pretrained(
            model_id,
            dtype=torch.float16 if device == "cuda" else torch.float32
        ).to(device)

        model_type = "seq2seq"

    else:
        if device == "cuda" and use_4bit:
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16
            )

            model = AutoModelForCausalLM.from_pretrained(
                model_id,
                quantization_config=bnb_config,
                device_map="auto"
            )
        else:
            model = AutoModelForCausalLM.from_pretrained(
                model_id,
                dtype=torch.float16 if device == "cuda" else torch.float32
            ).to(device)

        model_type = "causal"

    model.eval()
    return tokenizer, model, model_type, device

In [ ]:
def evaluate(model_id, batch_size=8):
    print(f"Evaluating {model_id}")

    tokenizer, model, model_type, device = load(model_id)

    prompts = []
    answers = []

    for qaPair in tqdm(gsm8k, desc="Preparing prompts"):
        question = qaPair["question"]
        answer = qaPair["answer"].split("####")[-1].strip().replace(",", "")

        prompt = few_shots + f"\n\nQ: {question}\nA:"
        prompts.append(prompt)
        answers.append(answer)

    outputs = []

    for i in tqdm(range(0, len(prompts), batch_size), desc="Generating answers"):
        batch_prompts = prompts[i:i + batch_size]

        inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True
        )

        if model_type == "seq2seq":
            inputs = inputs.to(device)

        else:
            inputs = inputs.to(model.device)

        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=128,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id
            )

        if model_type == "causal":
            input_length = inputs["input_ids"].shape[1]
            generated_ids = generated_ids[:, input_length:]

        batch_texts = tokenizer.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )

        outputs.extend(batch_texts)

    correct = 0

    for generated_text, answer in tqdm(
        zip(outputs, answers),
        total=len(answers),
        desc="Checking accuracy"
    ):
        pred = extract_final_answer(generated_text)

        if pred == answer:
            correct += 1

    total = len(gsm8k)
    accuracy = correct / total

    print(f"Accuracy: {accuracy:.2%}")
    return accuracy